In [1]:
import re, json, glob, os

In [73]:
year = 2015
txt_dir = f'../data/raw_text/NLE_{year}/'

all_files = glob.glob(txt_dir + '/*')

In [74]:
# look for this year and "Key" in the filename
key_file = [f for f in all_files if str(year) in f and "Key" in f][0]
print(key_file)

# all other files of this year which are not they answer key
exam_files = [f for f in all_files if str(year) in f and not f == key_file]
print(exam_files)


../data/raw_text/NLE_2015/2015 Answer Key and Translations rev.txt
['../data/raw_text/NLE_2015/2015 Introduction to Latin Exam rev 1.txt', '../data/raw_text/NLE_2015/2015 Latin I Exam FINAL rev_doc.txt', '../data/raw_text/NLE_2015/2015 Latin II Exam FINAL rev.txt', '../data/raw_text/NLE_2015/2015 Latin III Exam FINAL rev.txt', '../data/raw_text/NLE_2015/2015 Latin III-IV Poetry Exam FINAL rev.txt', '../data/raw_text/NLE_2015/2015 Latin III-IV Prose Exam FINAL rev.txt', '../data/raw_text/NLE_2015/2015 Latin V-VI Exam FINAL rev.txt']


answer key file

In [75]:
with open(key_file, 'r') as f:
    key_txt = f.read()


In [27]:
re.match(rf'(\*\*)?{levels[0]}(\*\*)?\s+', '**Introduction to Latin**\n')

<re.Match object; span=(0, 26), match='**Introduction to Latin**\n'>

In [76]:
# look for segments of the form:
# [YEAR] NATIONAL LATIN EXAM  <--- year given by us
# ANSWER KEYS AND TRANSLATIONS 
# [NAME OF LEVEL]               <--- we want to keep this
# 1. A/B/C/D [newlines/spaces]

# then we ignore questions with sentences (only keep multiple choice answers)
# also ignore the free text translations
# ignore the Copyright sections

# sometimes will have ** around the headers
if year == 2025:
    re_string = r'(\*\*)?\d{4} NATIONAL LATIN EXAM\sANSWER KEYS AND TRANSLATIONS(\*\*)?\s+(\*\*)?([\w\s]+)(\*\*)?\s+'
    chunks = re.split(re_string, key_txt)

# other years are simpler; search directly for the chunks we want
else:
    levels = [
        'Introduction to Latin', # may have ** surrounding it
        'Latin I',
        'Latin II',
        'Latin III',
        'Latin III-IV Prose',
        'Latin III-IV Poetry',
        'Latin V-VI'
    ]
    # put capture groups around each level
    #re_string = r'|'.join([rf'(\*\*)?({level})(\*\*)?\s*' for level in levels])

    # levels should be in order 
    chunks = []
    curr_level_idx = 0 # index of the current level we are looking for 
    curr_level = levels[curr_level_idx]
    for line in key_txt.split('\n'):
        # add back the \n so regex matches correctly 
        line = line + '\n'
        if re.match(rf'(\*\*)?{curr_level}(\*\*)?\s+', line):
            print(f'matched {curr_level}')
            chunks.append(line)
            curr_level_idx += 1

            
            curr_level = levels[curr_level_idx] if curr_level_idx < len(levels) else None
            chunks.append('')
        else:
            if chunks:
                chunks[-1] += '\n' + line
            else:
                chunks.append(line)



matched Introduction to Latin
matched Latin I
matched Latin II
matched Latin III
matched Latin III-IV Prose
matched Latin III-IV Poetry
matched Latin V-VI


In [13]:
re_string

'(\\*\\*)?(Introduction to Latin)(\\*\\*)?\\s*|(\\*\\*)?(Latin I)(\\*\\*)?\\s*|(\\*\\*)?(Latin II)(\\*\\*)?\\s*|(\\*\\*)?(Latin III)(\\*\\*)?\\s*|(\\*\\*)?(Latin III-IV Prose)(\\*\\*)?\\s*|(\\*\\*)?(Latin III-IV Poetry)(\\*\\*)?\\s*|(\\*\\*)?(Latin V-VI)(\\*\\*)?\\s*'

In [14]:
print(re.match(re_string, key_txt))

<re.Match object; span=(0, 26), match='**Introduction to Latin**\n'>


In [77]:
# remove any ** and empty strings in the chunks
chunks = [c for c in chunks if c != '' and c != '**' and c is not None]

In [78]:
chunks

['2015 NATIONAL LATIN EXAM\n\nANSWER KEYS AND TRANSLATIONS\n\n\n',
 'Introduction to Latin\n',
 '\n1. C  5. B  9. B  13. A  17. C  21. B  25. B  29. D  33. D  37. C\n\n2. D  6. C  10. C  14. D  18. B  22. D  26. A  30. B  34. C  38. A\n\n3. C  7. B  11. B  15. A  19. D  23. B  27. A  31. C  35. A  39. D\n\n4. B  8. A  12. D  16. B  20. D  24. C  28. A  32. A  36. B  40. C\n\n\n\n\n\nThe German slave flees out of the forum. The master and his two sons chase the slave. The master takes hold of the slave. The master fights with the slave. A crowd sees and surrounds the fight. The slave is afraid of the crowd. The slave tries to escape and runs between the two boys. The slave bumps into the boys accidentally and falls down onto the ground. "You dare to bump into my sons," the master shouts. "I ask for the death penalty for you because you are hurting my sons." "Father," one son says, "The German slave was hurting us accidentally. Don\'t kill the slave. The slave is valuable. Sell the slave

In [79]:
chunks = chunks[1:]
chunks

['Introduction to Latin\n',
 '\n1. C  5. B  9. B  13. A  17. C  21. B  25. B  29. D  33. D  37. C\n\n2. D  6. C  10. C  14. D  18. B  22. D  26. A  30. B  34. C  38. A\n\n3. C  7. B  11. B  15. A  19. D  23. B  27. A  31. C  35. A  39. D\n\n4. B  8. A  12. D  16. B  20. D  24. C  28. A  32. A  36. B  40. C\n\n\n\n\n\nThe German slave flees out of the forum. The master and his two sons chase the slave. The master takes hold of the slave. The master fights with the slave. A crowd sees and surrounds the fight. The slave is afraid of the crowd. The slave tries to escape and runs between the two boys. The slave bumps into the boys accidentally and falls down onto the ground. "You dare to bump into my sons," the master shouts. "I ask for the death penalty for you because you are hurting my sons." "Father," one son says, "The German slave was hurting us accidentally. Don\'t kill the slave. The slave is valuable. Sell the slave and keep the money." "Yes," the master replies, "You are clever."\

In [80]:
questions = re.findall(r'(\d+)\. *(A|B|C|D)\s+', chunks[1])
questions

[('1', 'C'),
 ('5', 'B'),
 ('9', 'B'),
 ('13', 'A'),
 ('17', 'C'),
 ('21', 'B'),
 ('25', 'B'),
 ('29', 'D'),
 ('33', 'D'),
 ('37', 'C'),
 ('2', 'D'),
 ('6', 'C'),
 ('10', 'C'),
 ('14', 'D'),
 ('18', 'B'),
 ('22', 'D'),
 ('26', 'A'),
 ('30', 'B'),
 ('34', 'C'),
 ('38', 'A'),
 ('3', 'C'),
 ('7', 'B'),
 ('11', 'B'),
 ('15', 'A'),
 ('19', 'D'),
 ('23', 'B'),
 ('27', 'A'),
 ('31', 'C'),
 ('35', 'A'),
 ('39', 'D'),
 ('4', 'B'),
 ('8', 'A'),
 ('12', 'D'),
 ('16', 'B'),
 ('20', 'D'),
 ('24', 'C'),
 ('28', 'A'),
 ('32', 'A'),
 ('36', 'B'),
 ('40', 'C')]

In [81]:
# now create a dictionary
# {level: {question_number: A/B/C/D}}

answer_keys = {}
curr_header = None
for i in range(len(chunks)):

    curr_chunk = chunks[i]
    if curr_chunk is None: 
        continue

    # otherwise, it will be a header OR a list of answers 
    # if there are numbers, it is a list of answers
    if re.match(r'\s*\d+', curr_chunk):
        # find all QA pairs (only letter answers)
        questions = re.findall(r'(\d+)\. *(A|B|C|D)\s+', curr_chunk)
        for (q, a) in questions:
            answer_keys[curr_header][q] = a
        

    else:
        curr_header = curr_chunk.strip()
        curr_header = curr_header.replace('**', '')
        answer_keys[curr_header] = {}

In [82]:
answer_keys

{'Introduction to Latin': {'1': 'C',
  '5': 'B',
  '9': 'B',
  '13': 'A',
  '17': 'C',
  '21': 'B',
  '25': 'B',
  '29': 'D',
  '33': 'D',
  '37': 'C',
  '2': 'D',
  '6': 'C',
  '10': 'C',
  '14': 'D',
  '18': 'B',
  '22': 'D',
  '26': 'A',
  '30': 'B',
  '34': 'C',
  '38': 'A',
  '3': 'C',
  '7': 'B',
  '11': 'B',
  '15': 'A',
  '19': 'D',
  '23': 'B',
  '27': 'A',
  '31': 'C',
  '35': 'A',
  '39': 'D',
  '4': 'B',
  '8': 'A',
  '12': 'D',
  '16': 'B',
  '20': 'D',
  '24': 'C',
  '28': 'A',
  '32': 'A',
  '36': 'B',
  '40': 'C'},
 'Latin I': {'1': 'D',
  '5': 'B',
  '9': 'B',
  '13': 'A',
  '17': 'C',
  '21': 'A',
  '25': 'A',
  '29': 'D',
  '33': 'D',
  '37': 'B',
  '2': 'A',
  '6': 'A',
  '10': 'A',
  '14': 'D',
  '18': 'A',
  '22': 'B',
  '26': 'D',
  '30': 'D',
  '34': 'C',
  '38': 'D',
  '3': 'C',
  '7': 'C',
  '11': 'D',
  '15': 'B',
  '19': 'B',
  '23': 'D',
  '27': 'C',
  '31': 'B',
  '35': 'C',
  '39': 'B',
  '4': 'D',
  '8': 'D',
  '12': 'C',
  '16': 'A',
  '20': 'B',
  '24'

In [83]:
# save the answer keys
if not os.path.exists(f'../data/semi_structured/NLE_{year}'):
    os.makedirs(f'../data/semi_structured/NLE_{year}')
with open(f'../data/semi_structured/NLE_{year}/answer_keys_{year}.json', 'w') as f:
    json.dump(answer_keys, f, indent=4)

process the individual exams

In [58]:
def get_exam_name(first_four_lines):
    # should be one of the first four lines,
    # the one that is NOT either 
    # 1. [year] ACL/NJCL NATIONAL LATIN EXAM
    # 2. EXAM [letter]
    # 3. CHOOSE THE BEST ANSWER...

    bad_starts = [
        f'{year} ACL/NJCL NATIONAL LATIN EXAM',
        f'**{year} ACL/NJCL',
        'ACL/NJCL NATIONAL LATIN EXAM',
        f'EXAM',
        'CHOOSE THE BEST ANSWER'
    ]
    re_start = r'[IV]+ EXAM'
    for line in first_four_lines:
        if (not any(line.startswith(bad_start) for bad_start in bad_starts) and
            re.match(re_start, line) is None):
            return line.strip()
    return None


In [60]:
def is_question_line(i, line, all_lines) -> bool:
    if not re.match(r'\d+', line):
        return False 
    
    # need to differentiate between line annotations/definitions and questions
    # line annotations won't have multiple choice answers
    # but questions could be multi-line, so must check next line

    # if this line or one of the next 4 lines has a match for D) then it is a question 
    if re.search(r'D\)', line):
        print("  matched single-line question")
        return True 
    elif any(re.search(r'D\)', all_lines[i+j]) for j in range(1,5)):
        print("  matched multi-line question")
        return True 
    
    print("  not a question line")
    return False 


In [61]:
def is_multi_line_question(i, line, all_lines) -> bool:
    '''assumes this is a question line, checks if it is multi-line'''

    # if D) is in this line, then it is not multi-line
    if re.search(r'D\)', line):
        return False 
    
    # if the next line has a A/B/C/D), then it is multi-line
    if re.search(r'[A-D]\)', all_lines[i+1]):
        return True 
    return False 



In [62]:
def get_multi_line_question_text(i, line, all_lines) -> str:
    '''assumes this is a multi-line question, returns the question text'''
    text = ''
    
    # add all next lines until after we've seen D
    answer_choices_left = ['A)', 'B)', 'C)', 'D)']

    for j in range(i, len(all_lines)):
        choices_found = re.findall(r'[A-D]\)', all_lines[j])
        if choices_found:
            answer_choices_left = [c for c in answer_choices_left if c not in choices_found]
        
        text += ' ' + all_lines[j].strip() 

        if len(answer_choices_left) == 0:
            break
    return text


In [68]:
def process_exam(exam_file):
    # first, do multiple choice questions

    with open(exam_file, 'r') as f:
        exam_lines = f.readlines()
    print(exam_file)

    # first line is the level (exam name)
    level = get_exam_name(exam_lines[:4])
    if level is None:
        raise ValueError(f'Could not find exam name in {exam_file}')

    this_exam = {}
    story_lines = []
    story_num = -1

    # if this is a reading comprehension exam,
    # then the story lines will come before the first question
    if "reading comprehension" in level.lower():
        getting_story_lines = True
        story_num = 0
    else:
        getting_story_lines = False

    # search for first line starting with a number (should be 1)
    # this is the first question
    for i, line in enumerate(exam_lines[5:]):
        print(i, line)
        if is_question_line(i, line, exam_lines[5:]):
            print("  question line")
            
            if is_multi_line_question(i, line, exam_lines[5:]):
                print("    multi-line question")
                question_line = get_multi_line_question_text(i, line, exam_lines[5:])
            else:
                print("    single-line question")
                question_line = line.strip()
            
            match = re.search(r'(\d+)\. (.*)', question_line)
            if match:
                question_number = match.group(1)
                question_text = match.group(2)
            else:
                print(f"No match found for question line: {question_line}")
                continue
            
            # we must be done reading the story lines, so save them to the dict 
            if getting_story_lines:
                getting_story_lines = False 
                story_text = '\n'.join(story_lines)
                this_exam[f'passage_{story_num}'] = story_text
                this_exam[f'passage_questions_{story_num}'] = [] # all questions that follow the passage
            
            
            # find the answer line
            match = re.search(r'(A\).*?)(B\).*?)(C\).*?)(D\).*)', question_line)
            if match:
                a = match.group(1)
                b = match.group(2)
                c = match.group(3)
                d = match.group(4)
                
                # get question text only, without number and answer choices
                question_text = question_text.split(a)[0].strip()
                
                a = a[2:].strip() # remove A) from start
                b = b[2:].strip() # remove B) from start
                c = c[2:].strip() # remove C) from start
                d = d[2:].strip() # remove D) from start

                # now we have A, B, C, D
                this_exam[question_number] = {
                    'question': question_text,
                    'A': a,
                    'B': b,
                    'C': c,
                    'D': d
                }

                # if passage is in the dict, this question is a passage question
                if f'passage_{story_num}' in this_exam:
                    this_exam[f'passage_questions_{story_num}'].append(question_number)


        elif line.startswith('READ THE PASSAGE AND ANSWER THE QUESTIONS.'):
            getting_story_lines = True
            story_num += 1
            story_lines = []
        
        # title of a new story
        elif not getting_story_lines and line.isupper():
            getting_story_lines = True
            story_num += 1
            story_lines = []
            story_lines.append(line.strip())

        elif getting_story_lines:
            print("    story line")
            story_lines.append(line.strip())

    return {level: this_exam}



In [84]:
exam_dict = {}
for exam_file in exam_files:
    exam_dict.update(process_exam(exam_file))


../data/raw_text/NLE_2015/2015 Introduction to Latin Exam rev 1.txt
0 1. Which animal has four legs? A) piscis B) avis C) equus D) homō

  matched single-line question
  question line
    single-line question
1 2. What animal is associated with the founding of Rome? A) elephant B) rabbit C) goose D) wolf

  matched single-line question
  question line
    single-line question
2 3. Which deity do the symbols on this coin represent? A) Juno B) Venus C) Minerva D) Vesta

  matched single-line question
  question line
    single-line question
3 4. The Latin phrase ita vērō is the opposite of A) bene B) minimē C) salvē D) grātiās

  matched single-line question
  question line
    single-line question
4 5. What main room of a Roman house had an impluvium and compluvium as well as a shrine to the household gods? A) cubiculum B) ātrium C) culīna D) trīclīnium

  matched single-line question
  question line
    single-line question
5 6. What is the Latin for "in God we hope," the motto of Brow

In [86]:
# save to file 
with open(f'../data/semi_structured/NLE_{year}/exams_{year}.json', 'w') as f:
    json.dump(exam_dict, f, indent=4)

In [85]:
len(exam_files)

7

In [64]:
idx = 0
exam_files[idx]

'../data/raw_text/NLE_2020/2020 Latin I Exam.txt'

In [65]:
exam_dict = process_exam(exam_files[idx])

0 1. Ubi est amīcus meus? A) Who B) Where C) How D) Why

  matched single-line question
  question line
    single-line question
1 

2 2. Ad vīllam saepe ambulābāmus. A) I was walking B) You were walking C) We were walking D) They were walking

  matched single-line question
  question line
    single-line question
3 

4 3. Puellae in agrīs sunt sed puerī in silvīs sunt. A) but B) when C) and D) or

  matched single-line question
  question line
    single-line question
5 

6 4. Nōlīte intrare in vīllam nostram! A) on B) near C) out of D) into

  matched single-line question
  question line
    single-line question
7 

8 5. Ego multās grātiās tibi agō. What am I doing? A) insulting you B) giving an order C) expressing anger D) saying thank you

  matched single-line question
  question line
    single-line question
9 

10 6. Vōs omnēs in agrīs labōrāvistis. A) have worked B) were working C) are working D) will work

  matched single-line question
  question line
    single-line questio

In [48]:
line = "39. How does the sailor solve Aelia's problem (lines 9-12)? A) He carries her to the front. B) He places her onto the wall."

In [52]:
re.findall(r'[A-D]\)', line)

['A)', 'B)']

In [66]:
exam_dict

{'LATIN I': {'1': {'question': 'Ubi est amīcus meus?',
   'A': 'Who',
   'B': 'Where',
   'C': 'How',
   'D': 'Why'},
  '2': {'question': 'Ad vīllam saepe ambulābāmus.',
   'A': 'I was walking',
   'B': 'You were walking',
   'C': 'We were walking',
   'D': 'They were walking'},
  '3': {'question': 'Puellae in agrīs sunt sed puerī in silvīs sunt.',
   'A': 'but',
   'B': 'when',
   'C': 'and',
   'D': 'or'},
  '4': {'question': 'Nōlīte intrare in vīllam nostram!',
   'A': 'on',
   'B': 'near',
   'C': 'out of',
   'D': 'into'},
  '5': {'question': 'Ego multās grātiās tibi agō. What am I doing?',
   'A': 'insulting you',
   'B': 'giving an order',
   'C': 'expressing anger',
   'D': 'saying thank you'},
  '6': {'question': 'Vōs omnēs in agrīs labōrāvistis.',
   'A': 'have worked',
   'B': 'were working',
   'C': 'are working',
   'D': 'will work'},
  '7': {'question': 'Feminae multōs librōs lēgērunt. How many books were read?',
   'A': 'some',
   'B': 'all',
   'C': 'few',
   'D': 'many